# 20. Parent-Document and Multi-Vector Retrieval

**Complexity:** 3/5

## Overview

Every RAG pipeline so far embedded **the same text it returned to the LLM**. This notebook decouples the two: we search over one representation and hand the LLM another.

### The Problem

Chunk size is a lose-lose trade-off:
- **Small chunks** (200-400 chars) produce focused embeddings that match queries precisely, but give the LLM too little context to answer
- **Large chunks** (2000+ chars) carry enough context, but their embedding averages many topics and matches no query well
- **Raw text is not what users ask**: a chunk of code and docs rarely looks like the question it answers

### The Solution

Two retrievers that separate *what you search* from *what you return*:

1. **Parent-Document Retriever (small-to-big)**: embed small child chunks, but return the larger parent chunk they came from
2. **Multi-Vector Retriever**: embed several LLM-generated representations of each chunk (a **summary** and **hypothetical questions** it answers), and return the original chunk

Hypothetical questions are the mirror image of HyDE (notebook 06): HyDE turns the *query* into a fake answer at query time; here we turn each *chunk* into fake questions at indexing time, so question-shaped queries meet question-shaped vectors, at zero query-time cost.

### Pipeline

```
Parent-Document                         Multi-Vector

parent (2000 chars)                     chunk (1000 chars)
 ├─ child 1 ─► embed ─┐                  ├─ summary      ─► embed ─┐
 ├─ child 2 ─► embed ─┼─ vector store    ├─ question 1   ─► embed ─┼─ vector store
 └─ child 3 ─► embed ─┘                  └─ question 2-3 ─► embed ─┘
                  │ match child                       │ match any representation
                  ▼                                   ▼
          return parent (docstore)            return original chunk (docstore)
```

Both use a **docstore** (key-value store) keyed by `doc_id`: the vector store holds the searchable vectors, the docstore holds the text returned to the LLM.

### When to Use

| Technique | Use when | Cost |
|-----------|----------|------|
| **Parent-Document** | Answers need surrounding context (procedures, long code examples) | More embeddings (small chunks), larger prompts |
| **Multi-Vector** | Users ask questions in words the text doesn't use; chunks are dense (code, tables) | 1-2 LLM calls per chunk at indexing time |


## 1. Setup and Imports

In [ ]:
import json
import sys
import time
import uuid
from pathlib import Path

# Add parent directory to path for imports
sys.path.append(str(Path("../..").resolve()))

import pandas as pd
from langchain_classic.retrievers import MultiVectorRetriever, ParentDocumentRetriever
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.stores import InMemoryStore
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

from shared.config import (
    verify_api_key,
    CACHE_DIR,
    DEFAULT_MODEL,
    DEFAULT_TEMPERATURE,
    OPENAI_EMBEDDING_MODEL,
)
from shared.loaders import load_and_split
from shared.prompts import (
    CHUNK_SUMMARY_PROMPT,
    HYPOTHETICAL_QUESTIONS_PROMPT,
    RAG_PROMPT_TEMPLATE,
)
from shared.utils import format_docs, print_section_header

# Verify API key
verify_api_key()

print("All imports successful")
print(f"Using model: {DEFAULT_MODEL}")
print(f"Using embeddings: {OPENAI_EMBEDDING_MODEL}")

## 2. Load Documents and Build the Baseline

The baseline is standard RAG: 1000-character chunks, embedded and returned as-is.

All indexes in this notebook use `InMemoryVectorStore` and `InMemoryStore`, rebuilt on each run. Embedding a few hundred chunks costs a fraction of a cent; in production you would use a persistent vector store and docstore.

In [ ]:
print_section_header("Loading Documents")

# Full pages (for parent-document) and standard chunks (baseline and multi-vector)
original_docs, chunks = load_and_split(chunk_size=1000, chunk_overlap=200)

print(f"\n{len(original_docs)} documents, {len(chunks)} chunks")

embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)
llm = ChatOpenAI(model=DEFAULT_MODEL, temperature=DEFAULT_TEMPERATURE)

K = 4

baseline_store = InMemoryVectorStore.from_documents(chunks, embeddings)
baseline_retriever = baseline_store.as_retriever(search_kwargs={"k": K})

print("Baseline retriever ready (1000-char chunks)")

## 3. Parent-Document Retriever (Small-to-Big)

- **Child splitter** (400 chars): what gets embedded and matched
- **Parent splitter** (2000 chars): what gets returned. Without a parent splitter, the whole source document is the parent, too large for web pages like ours

The retriever fetches the top child matches, maps each to its `doc_id`, and returns the unique parents. We ask for more children than needed (`k=12`) because several children often share a parent.

In [ ]:
print_section_header("Building Parent-Document Retriever")

child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)

parent_retriever = ParentDocumentRetriever(
    vectorstore=InMemoryVectorStore(embeddings),
    docstore=InMemoryStore(),
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 12},
)

start = time.time()
parent_retriever.add_documents(original_docs)

num_parents = len(list(parent_retriever.docstore.yield_keys()))
num_children = len(parent_retriever.vectorstore.store)
print(f"\nIndexed {num_children} child chunks → {num_parents} parents in {time.time() - start:.1f}s")

In [ ]:
query = "How do I drop the oldest messages so the history fits in the context window?"

child_hits = parent_retriever.vectorstore.similarity_search(query, k=2)
parents = parent_retriever.invoke(query)[:K]

print(f"Query: {query}\n")
print("TOP CHILD MATCH (what was searched):")
print("-" * 80)
print(f"[{len(child_hits[0].page_content)} chars] {child_hits[0].page_content}\n")
print("RETURNED PARENT (what the LLM sees):")
print("-" * 80)
print(f"[{len(parents[0].page_content)} chars] {parents[0].page_content[:800]}...")

## 4. Multi-Vector Retriever

For each 1000-char chunk we generate:
- a **summary** (2-3 sentences naming the APIs it covers)
- **3 hypothetical questions** it answers, via `with_structured_output` so we get a clean list instead of parsing text

Each representation is embedded with metadata `doc_id` pointing to the original chunk in the docstore. Generations are cached in `data/cache/` so re-running the notebook does not repeat the LLM calls.

In [ ]:
print_section_header("Generating Chunk Representations")


class HypotheticalQuestions(BaseModel):
    questions: list[str] = Field(description="Questions the text answers")


summary_chain = CHUNK_SUMMARY_PROMPT | llm | StrOutputParser()
questions_chain = HYPOTHETICAL_QUESTIONS_PROMPT | llm.with_structured_output(HypotheticalQuestions)

cache_path = CACHE_DIR / f"multivector__{DEFAULT_MODEL}__{len(chunks)}.json"

if cache_path.exists():
    representations = json.loads(cache_path.read_text())
    print(f"Loaded {len(representations)} cached representations from {cache_path.name}")
else:
    print(f"Generating summaries and questions for {len(chunks)} chunks...")
    start = time.time()
    inputs = [{"chunk": c.page_content} for c in chunks]
    summaries = summary_chain.batch(inputs, config={"max_concurrency": 8})
    questions = questions_chain.batch(
        [{**i, "num_questions": 3} for i in inputs], config={"max_concurrency": 8}
    )
    representations = [
        {"summary": s, "questions": q.questions} for s, q in zip(summaries, questions)
    ]
    cache_path.write_text(json.dumps(representations, indent=2))
    print(f"Generated {2 * len(chunks)} LLM calls in {time.time() - start:.1f}s")

example = 10
print("\nORIGINAL CHUNK:")
print("-" * 80)
print(chunks[example].page_content[:400] + "...")
print("\nSUMMARY:")
print("-" * 80)
print(representations[example]["summary"])
print("\nHYPOTHETICAL QUESTIONS:")
print("-" * 80)
for q in representations[example]["questions"]:
    print(f"  • {q}")

In [ ]:
print_section_header("Building Multi-Vector Retriever")

ID_KEY = "doc_id"
doc_ids = [str(uuid.uuid4()) for _ in chunks]

multi_vector_retriever = MultiVectorRetriever(
    vectorstore=InMemoryVectorStore(embeddings),
    docstore=InMemoryStore(),
    id_key=ID_KEY,
    search_kwargs={"k": 12},  # Over-fetch: several vectors can point to the same chunk
)

# Searchable representations → vector store, originals → docstore
sub_docs = []
for doc_id, rep in zip(doc_ids, representations):
    texts = [rep["summary"], *rep["questions"]]
    sub_docs += [Document(page_content=t, metadata={ID_KEY: doc_id}) for t in texts]

multi_vector_retriever.vectorstore.add_documents(sub_docs)
multi_vector_retriever.docstore.mset(list(zip(doc_ids, chunks)))

print(f"Indexed {len(sub_docs)} vectors for {len(chunks)} chunks")

## 5. Quantitative Comparison

Same method as notebook 19: each question has **markers**, strings that only occur in the chunks that answer it. We use question-style queries only, the case multi-vector indexing targets.

Besides **Hit@4** and **MRR@4** we measure the **context size** sent to the LLM: parent-document retrieval finds answers partly because it returns more text, and that text is paid for on every query.

In [ ]:
EVAL_SET = [
    ("How do I limit how many requests per second my app sends to the model provider?", ["rate_limiter"]),
    ("How can I automatically condense old conversation history into a shorter version?", ["SummarizationMiddleware"]),
    ("How do I make the model reply with data that follows a schema?", ["with_structured_output"]),
    ("How do I get results for many inputs as soon as each one finishes?", ["batch_as_completed"]),
    ("How do I drop the oldest messages so the history fits in the context window?", ["trim_messages", "RemoveMessage"]),
    ("How do I switch between model providers at runtime without changing code?", ["configurable_model"]),
    ("How do I retry failed model calls automatically?", ["max_retries"]),
    ("How can I make the model think harder before answering?", ["reasoning_effort"]),
]

retrievers = {
    "Baseline (1000 chars)": baseline_retriever,
    "Parent-Document": parent_retriever,
    "Multi-Vector": multi_vector_retriever,
}


def first_relevant_rank(results, markers) -> int | None:
    # 1-based rank of the first chunk containing a marker, None if absent
    for rank, doc in enumerate(results, 1):
        if any(m in doc.page_content for m in markers):
            return rank
    return None


rows = []
for name, retriever in retrievers.items():
    for query, markers in EVAL_SET:
        results = retriever.invoke(query)[:K]
        rank = first_relevant_rank(results, markers)
        rows.append(
            {
                "retriever": name,
                "hit": rank is not None,
                "rr": 1 / rank if rank else 0.0,
                "context_chars": sum(len(d.page_content) for d in results),
            }
        )

print_section_header("Retrieval Comparison")
comparison = (
    pd.DataFrame(rows)
    .groupby("retriever")
    .agg(**{"Hit@4": ("hit", "mean"), "MRR@4": ("rr", "mean"), "avg context chars": ("context_chars", "mean")})
    .reindex(list(retrievers))
)
print(comparison.round(2).to_string())

**How to read it:** compare quality *and* context size. A parent-document hit is expected to cost roughly 2x the prompt tokens of the baseline; multi-vector returns the same 1000-char chunks as the baseline, so any gain there comes purely from better matching.

> With 8 queries, one query moves a score by 0.125: treat this as a sanity check. Notebook 16 shows how to evaluate answers end-to-end.

## 6. RAG Chains

In [ ]:
print_section_header("Comparing Answers")


def build_chain(retriever):
    return (
        {"context": RunnableLambda(lambda q: retriever.invoke(q)[:K]) | format_docs, "input": RunnablePassthrough()}
        | RAG_PROMPT_TEMPLATE
        | llm
        | StrOutputParser()
    )


chains = {name: build_chain(r) for name, r in retrievers.items()}

query = "How can I make the model think harder before answering?"
print(f"Query: {query}")

for name, chain in chains.items():
    print("\n" + "=" * 80)
    print(f"[{name}]")
    print("=" * 80)
    start = time.time()
    print(chain.invoke(query))
    print(f"\nTime: {time.time() - start:.2f}s")

## 7. Indexing Cost

In [ ]:
print_section_header("Indexing Cost")

cost = pd.DataFrame(
    [
        {"retriever": "Baseline (1000 chars)", "vectors": len(chunks), "llm_calls": 0},
        {"retriever": "Parent-Document", "vectors": num_children, "llm_calls": 0},
        {"retriever": "Multi-Vector", "vectors": len(sub_docs), "llm_calls": 2 * len(chunks)},
    ]
)
print(cost.to_string(index=False))
print("\nQuery-time cost is the same for all three: one embedding call, no LLM calls.")

## 8. Key Takeaways

### Summary

- **Separate what you search from what you return**: a vector store for matching, a docstore for the text the LLM reads
- **Parent-Document** fixes the chunk-size dilemma: precise small-chunk matching, full-context answers
- **Multi-Vector** indexes what users will *ask*, not only what the text *says*, and moves the LLM cost from query time (HyDE) to indexing time

### Cost-Benefit Analysis

| Aspect | Parent-Document | Multi-Vector |
|--------|-----------------|--------------|
| **Indexing cost** | More, smaller embeddings | 1-2 LLM calls per chunk |
| **Query latency** | Same | Same |
| **Prompt size** | Larger (parents) | Same as baseline |
| **Best for** | Context-hungry answers | Vocabulary mismatch, code, tables |

### Best Practices

1. **Always over-fetch** (`search_kwargs={"k": ...}`): several vectors map to the same document and are deduplicated
2. **Use a parent splitter** unless your source documents are short
3. **Persist the docstore** in production (`LocalFileStore`, Redis, a database), since `InMemoryStore` is lost on restart
4. **Cache LLM-generated representations**: they are the expensive part of re-indexing
5. **Combine with notebook 19**: multi-vector or parent retrieval as the first stage, then a cross-encoder reranker

### Related Techniques

- **Contextual retrieval** (notebook 12): prepends context to each chunk instead of indexing extra vectors
- **RAPTOR**: recursively clusters and summarizes chunks into a tree, and indexes every level for questions that span many chunks
- **Late chunking**: embeds the whole document with a long-context embedding model, then pools token embeddings per chunk so each chunk vector sees the full document

### Next Steps

- **Notebook 19**: add hybrid search and reranking on top
- **Notebook 16**: evaluate these retrievers end-to-end with RAGAS